# Train a language model from scratch

Every part of this model is hand-written: the BPE tokenizer, the transformer,
AdamW, the training loop, KV-cached generation, checkpointing. PyTorch supplies
tensors, autograd and BLAS — there is no `torch.nn.Linear`, no `torch.optim`,
no HuggingFace.

Run the cells in order. On a free Colab/Kaggle GPU the default configuration
trains in roughly **20–30 minutes** and produces a model that writes real
English sentences.

**Colab:** Runtime → Change runtime type → T4 GPU.


## 1. What hardware did we get?

This decides everything downstream — model size and how long to train.


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'No GPU — CPU only')
print('python', sys.version.split()[0])


## 2. Get the code and install


In [ ]:
!git clone --branch main-5h8bvh --single-branch \
    https://github.com/m9cherif/ai_from_zero.git 2>/dev/null || echo 'already cloned'
%cd ai_from_zero
!pip install -q -r requirements.txt

import torch
from myai.core.device import setup, describe
print(describe(setup('auto')))


## 3. Download the corpus

About 100 public-domain books (Melville, Austen, Dickens, Dostoevsky, Tolstoy,
Shakespeare, Milton, Plato, Darwin) plus WikiText-2 — roughly **25 million tokens**.

Validation documents are held out *whole*, and the script measures its own
train/val overlap afterwards. That number should print as 0%.


In [ ]:
!python scripts/fetch_corpus.py


## 4. Build the tokenizer

Byte-pair encoding trained on the corpus. Takes a couple of minutes and must
report `lossless decode: True` — if it doesn't, stop here.


In [ ]:
!python scripts/build_tokenizer.py --type bpe --vocab-size 4096 \
    --data 'data/train/*.txt'


## 5. Choose a size

| Model | `--d-model` | `--n-layers` | Trains in | Result |
|---|---|---|---|---|
| **8.3M** (recommended) | 320 | 6 | ~25 min on T4 | perplexity 47.8, writes English |
| 25M | 512 | 8 | ~1.5 h | better, needs more data to shine |
| 915M (`--preset xl`) | — | — | days | see the last section |

Bigger is not better here. With 25M tokens of text, a model much past ~10M
parameters is undertrained: it learns which words are common and stops. That
is measured, not assumed — see the final section.


In [ ]:
# The corpus is tokenized once into a memory-mapped array, so epochs after
# the first cost nothing extra. --device auto picks the GPU with the most
# free memory, or falls back to CPU.
!python scripts/train.py \
    --d-model 320 --n-layers 6 --seq-len 256 \
    --data data/train --val-data data/val --cache-dir output/tokens \
    --steps 6500 --batch-size 16 --lr 6e-4 --dropout 0.05 \
    --flash --device auto --log-every 100 --save-every 500


## 6. Score it on text it has never seen

Perplexity against a uniform baseline of 4,096 (the vocabulary size). Anything
near 4,096 means the model learned nothing; the reference run reaches **47.8**.


In [ ]:
!python scripts/evaluate.py --data data/val --max-batches 200


## 7. Talk to it

This is a *base* model: it continues text, it does not answer questions.
`"To be, or not"` works; `"What is the capital of France?"` does not.


In [ ]:
!python scripts/chat.py --max-tokens 60 --prompt \
    'The old man walked into the' \
    'She said that' \
    'It was the best of'


## 8. Keep the model

Colab and Kaggle delete everything when the runtime stops. Save the checkpoint
somewhere durable before you close the tab.


In [ ]:
# Google Drive (Colab)
# from google.colab import drive; drive.mount('/content/drive')
# !cp output/checkpoints/checkpoint_latest.pt /content/drive/MyDrive/

# Or download it directly
# from google.colab import files; files.download('output/checkpoints/checkpoint_latest.pt')

!ls -la output/checkpoints/


### Optional: a self-contained web page

Quantizes the model to int8 and inlines it, the tokenizer and a JavaScript
port of the forward pass into one HTML file. It runs with no server — open it
anywhere. Works up to roughly 12M parameters before the file gets unwieldy.


In [ ]:
!python scripts/export_web.py --out web/model.json
!python scripts/build_web.py --out web/marginalia.html
# from google.colab import files; files.download('web/marginalia.html')


## 9. About the 915M model

`--preset xl` is 914,729,472 parameters. Whether it fits depends entirely on
the optimizer, because AdamW keeps two fp32 moments per parameter:

```
AdamW   weights 3.7 + grads 3.7 + m 3.7 + v 3.7 = 14.6 GB
SGD     weights 3.7 + grads 3.7               =  7.3 GB
```

So on a 16 GB T4 it trains only with SGD and gradient checkpointing; on an
A100 40 GB it trains with AdamW. **Inference is far cheaper** — weights only,
~3.7 GB — so a GPU that cannot train it can still serve it.

Be clear-eyed about the payoff. Trained for 840 steps on this corpus it reached
perplexity **1,022.8** and produced *"the his of and the and with to had he"*,
while the 8.3M model reached **47.8** and wrote real sentences. Its loss sat at
6.93 across eight measurements spanning a sevenfold range of training — a flat
line. Filling a billion parameters takes roughly 18 billion tokens; this corpus
has 25 million.

If you want to try it anyway:


In [ ]:
# Needs a 16 GB+ GPU. Add --optimizer adamw instead if you have 40 GB.
# !python scripts/train.py --preset xl \
#     --data data/train --val-data data/val --cache-dir output/tokens \
#     --steps 20000 --batch-size 8 --lr 0.02 \
#     --optimizer sgd --grad-checkpoint --no-save-optimizer \
#     --flash --device auto --log-every 50


---

**Repository:** [m9cherif/ai_from_zero](https://github.com/m9cherif/ai_from_zero)
(branch `main-5h8bvh`) · 171 tests · `docs/SCALING.md` has the measured basis
for every number quoted here.
